# 01 — Error analysis

Exploration of the classification errors produced by the four models.
This notebook is **exploratory only**: everything reproducible lives in
`src/` and is regenerated by `bash run_all.sh`. Run it from the `dl-scrna`
conda environment after a full run.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="ticks", context="notebook")
RESULTS = Path("../results")

leaderboard = pd.read_csv(RESULTS / "tables" / "leaderboard.csv")
per_class = pd.read_csv(RESULTS / "tables" / "per_class_f1_test.csv", index_col=0)
confusions = pd.read_csv(RESULTS / "tables" / "top_confusions.csv")
leaderboard

## Per-class performance on the held-out clusters

Only labels present in the test set are meaningfully scored here.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
per_class.plot.bar(ax=ax)
ax.set_ylabel("test F1")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
plt.xticks(rotation=45, ha="right")
plt.show()

## Which classes are confused, and does it make biological sense?

Expected confusions on PBMCs: CD4 vs CD8 T cells, NK vs cytotoxic T
cells, classical vs non-classical monocytes — closely related lineages
that share most of their expression programme.

In [ ]:
for model, sub in confusions.groupby("model"):
    top = sub.nlargest(5, "n_cells")
    print(f"\n== {model} ==")
    print(top.to_string(index=False))

## Where do errors sit on the UMAP?

Misclassified test cells should concentrate at cluster borders between
related lineages if the errors are biologically coherent.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad(RESULTS / "processed" / "pbmc3k_prepared.h5ad")
best = leaderboard.sort_values("val_macro_f1", ascending=False).iloc[0]["model"]
z = np.load(RESULTS / "predictions" / f"{best}_seed42.npz")
adata.obs["pred"] = z["y_pred"].astype(str)
adata.obs["correct"] = (z["y_pred"] == adata.obs["cell_type"].cat.codes.to_numpy())
adata.obs["correct"] = np.where(adata.obs["split"] == "test", adata.obs["correct"], "n/a")
sc.pl.umap(adata, color=["cell_type", "correct"], wspace=0.4)